In [1]:
# Setup
import sys
sys.path.append('..')

from src.multi_file_loader import MultiFileLoader
from src.context_preserver import ContextPreserver
from pathlib import Path
import json

print("✓ Imports successful")

✓ Imports successful


## Step 1: Load Sample Document

In [2]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Load documents first
loader = MultiFileLoader()

# Load from data folder
data_folder = Path("../data")

if data_folder.exists():
    documents = loader.load_all_documents(str(data_folder))
    print(f"\n✓ Loaded {len(documents)} documents")
    
    # Show first document structure
    if documents:
        doc = documents[0]
        print(f"\nFirst document: {doc['filename']} (type: {doc.get('type', 'unknown')})")
        text_count = len(doc.get('text_chunks', []))
        table_count = len(doc.get('tables', []))
        image_count = len(doc.get('images', []))
        # Some doc types store content directly instead of text_chunks
        if text_count == 0 and 'content' in doc:
            text_count = 1
        print(f"Elements: {text_count} text, {table_count} tables, {image_count} images")
        
    # Summary by type
    type_counts = {}
    for d in documents:
        t = d.get('type', 'unknown')
        type_counts[t] = type_counts.get(t, 0) + 1
    print("\nDocuments by type:")
    for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"  {t}: {c}")
else:
    print("⚠ No data folder found. Please add documents to ../data/")
    documents = []

Skipping code zBatchRollupReportIdGenerator.sj: Zero-byte file
Skipping code zCalculatedMeterEditor.sj: Zero-byte file
Skipping code zCFCalculation.sj: Zero-byte file
Skipping code zCFMigration.sj: Zero-byte file
Skipping code zCloseGroupScheduleDate.sj: Zero-byte file
Skipping code zContactDistributionList.sj: Zero-byte file
Skipping code zContactRelationship.sj: Zero-byte file
Skipping code zContactTypeEditor.sj: Zero-byte file
Skipping code zDataCondenser.sj: Zero-byte file
Skipping code zDataExchange.sj: Zero-byte file
Skipping code zDOTHazardEditor.sj: Zero-byte file
Skipping code zDurationFix.sj: Zero-byte file
Skipping code zEditFlowComputerMeters.sj: Zero-byte file
Skipping code zEditFlowComputerProduct.sj: Zero-byte file
Skipping code zEditReason.sj: Zero-byte file
Skipping code zEffeciencyDefinitionEditor.sj: Zero-byte file
Skipping code zEnterProductAlias.sj: Zero-byte file
Skipping code zExportDataEditor.sj: Zero-byte file
Skipping code zExternalVolumeDefaultValues.sj: Zero


✓ Loaded 22531 documents

First document: csh.js (type: code)
Elements: 1 text, 0 tables, 0 images

Documents by type:
  code: 11045
  text: 4836
  image: 1766
  docx: 1693
  excel: 1062
  html: 653
  structured_text: 623
  pdf: 457
  csv: 386
  doc_legacy: 10


## Step 2: Initialize Context Preserver

In [3]:
# Create context preserver
preserver = ContextPreserver()

print("✓ Context Preserver initialized")
print(f"  Documents ready: {len(documents)}")

✓ Context Preserver initialized
  Documents ready: 22531


## Step 3: Add Documents and Link Elements

In [4]:
# Process documents to create linked elements
if documents:
    skipped_no_chunks = 0
    for doc in documents:
        # Normalize: some doc types store 'content' instead of 'text_chunks'
        text_chunks = doc.get('text_chunks', [])
        if not text_chunks and 'content' in doc:
            # Convert bare-content docs (code, etc.) into a text_chunks list
            text_chunks = [{
                'element_id': f"{doc.get('file_id', 'unknown')}_text_0",
                'content': doc['content'],
                'type': 'text'
            }]
        
        if not text_chunks and not doc.get('tables', []) and not doc.get('images', []):
            skipped_no_chunks += 1
            continue
        
        preserver.add_document_elements(
            file_id=doc.get('file_id', ''),
            filename=doc.get('filename', ''),
            text_chunks=text_chunks,
            tables=doc.get('tables', []),
            images=doc.get('images', [])
        )
    
    print(f"\n✓ Processed {len(documents)} documents ({skipped_no_chunks} had no extractable chunks)")
    print(f"  Total elements: {len(preserver.all_elements)}")
    
    # Show element type distribution
    type_counts = {}
    for elem in preserver.all_elements:
        elem_type = elem['type']
        type_counts[elem_type] = type_counts.get(elem_type, 0) + 1
    
    print("\n  Element breakdown:")
    for elem_type, count in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"    {elem_type}: {count}")
else:
    print("⚠ No documents to process")


✓ Processed 22531 documents (2432 had no extractable chunks)
  Total elements: 123310

  Element breakdown:
    text: 109066
    table: 13027
    image: 831
    table_header: 386


## Step 4: Explore Hierarchical Paths

In [5]:
# Show hierarchy paths for first 5 elements
if preserver.all_elements:
    print("Sample Hierarchy Paths:\n")
    print("=" * 80)
    
    for idx, elem in enumerate(preserver.all_elements[:5], 1):
        print(f"\n{idx}. Element ID: {elem['element_id'][:16]}...")
        print(f"   Type: {elem['type']}")
        print(f"   Path: {elem['hierarchy_path']}")
        
        # Show snippet
        if 'content' in elem:
            text = elem['content'][:100] + "..." if len(elem['content']) > 100 else elem['content']
            print(f"   Content: {text}")
    
    print("\n" + "=" * 80)

Sample Hierarchy Paths:


1. Element ID: file_67d7d14bf00...
   Type: text
   Path: csh.js
   Content: /// <reference path="../Scripts/jquery.js" />
/// <reference path="../Scripts/MadCapUtilities.js" />...

2. Element ID: file_a5c4b0535fe...
   Type: text
   Path: Default.htm
   Content: [HTML file with no extractable text]

3. Element ID: file_08d8daf003a...
   Type: text
   Path: Default.js
   Content: // {{MadCap}} //////////////////////////////////////////////////////////////////
// Copyright: MadCa...

4. Element ID: file_cb04b2621ae...
   Type: text
   Path: Default_CSH.htm
   Content: [HTML file with no extractable text]

5. Element ID: file_9a480b787a1...
   Type: text
   Path: Search.htm
   Content: Search
Skip To Main Content
Account
Settings
Logout
placeholder
Account
Settings
Logout
Filter:
All ...



## Step 5: Examine Relationships

In [6]:
# Find elements with relationships
if preserver.all_elements:
    print("Elements with Relationships:\n")
    print("=" * 80)
    
    count = 0
    for elem in preserver.all_elements:
        has_text = len(elem.get('related_text_ids', [])) > 0
        has_images = len(elem.get('related_image_ids', [])) > 0
        has_tables = len(elem.get('related_table_ids', [])) > 0
        
        if has_text or has_images or has_tables:
            count += 1
            if count <= 3:  # Show first 3
                print(f"\n{count}. Element: {elem['type']} - {elem['hierarchy_path']}")
                print(f"   Related text chunks: {len(elem.get('related_text_ids', []))}")
                print(f"   Related images: {len(elem.get('related_image_ids', []))}")
                print(f"   Related tables: {len(elem.get('related_table_ids', []))}")
    
    print(f"\n\nTotal elements with relationships: {count} / {len(preserver.all_elements)}")
    print("=" * 80)

Elements with Relationships:


1. Element: text - Create User-Defined Fields.htm
   Related text chunks: 2
   Related images: 0
   Related tables: 0

2. Element: text - Create User-Defined Fields.htm
   Related text chunks: 2
   Related images: 0
   Related tables: 0

3. Element: text - Create User-Defined Fields.htm
   Related text chunks: 2
   Related images: 0
   Related tables: 0


Total elements with relationships: 91330 / 123310


## Step 6: Retrieve Related Elements

In [7]:
# Find an element with relationships and retrieve related content
if preserver.all_elements:
    # Find first element with related content
    target_elem = None
    for elem in preserver.all_elements:
        if (len(elem.get('related_text_ids', [])) > 0 or 
            len(elem.get('related_image_ids', [])) > 0 or 
            len(elem.get('related_table_ids', [])) > 0):
            target_elem = elem
            break
    
    if target_elem:
        print("Retrieving Related Elements:\n")
        print("=" * 80)
        print(f"\nSource Element: {target_elem['type']}")
        print(f"Path: {target_elem['hierarchy_path']}")
        
        # Get related elements (returns dict of ID lists by type)
        related = preserver.get_related_elements(target_elem['element_id'])
        
        related_text_ids = related.get('related_text', [])
        related_table_ids = related.get('related_tables', [])
        related_image_ids = related.get('related_images', [])
        
        total_related = len(related_text_ids) + len(related_table_ids) + len(related_image_ids)
        print(f"\nRelated element IDs found: {total_related}")
        print(f"  Text: {len(related_text_ids)}")
        print(f"  Tables: {len(related_table_ids)}")
        print(f"  Images: {len(related_image_ids)}")
        
        if related_text_ids:
            print(f"\nSample related text IDs:")
            for idx, rid in enumerate(related_text_ids[:5], 1):
                print(f"  {idx}. {rid}")
        
        print("\n" + "=" * 80)
    else:
        print("No elements with relationships found")

Retrieving Related Elements:


Source Element: text
Path: Create User-Defined Fields.htm

Related element IDs found: 2
  Text: 2
  Tables: 0
  Images: 0

Sample related text IDs:
  1. file_24f7e29e5455_text_1
  2. file_24f7e29e5455_text_2



## Step 7: Inspect Metadata Enrichment

In [8]:
# Show detailed metadata for one element
if preserver.all_elements:
    sample = preserver.all_elements[0]
    
    print("Detailed Element Metadata:\n")
    print("=" * 80)
    
    # Show all metadata fields
    metadata_fields = [
        'element_id',
        'file_id',
        'filename',
        'type',
        'content',
        'hierarchy_path',
        'page_number',
        'source_type',
        'element_index',
        'related_text_ids',
        'related_image_ids',
        'related_table_ids'
    ]
    
    for field in metadata_fields:
        if field in sample:
            value = sample[field]
            # Truncate lists
            if isinstance(value, list):
                if len(value) > 0:
                    print(f"{field}: {len(value)} items")
                    if len(value) <= 3:
                        for item in value:
                            print(f"  - {item[:40]}..." if len(str(item)) > 40 else f"  - {item}")
                else:
                    print(f"{field}: []")
            # Truncate long strings
            elif isinstance(value, str) and len(value) > 60:
                print(f"{field}: {value[:60]}...")
            else:
                print(f"{field}: {value}")
    
    print("\n" + "=" * 80)

Detailed Element Metadata:

element_id: file_67d7d14bf00c_text_0
file_id: file_67d7d14bf00c
filename: csh.js
type: text
content: /// <reference path="../Scripts/jquery.js" />
/// <reference...
hierarchy_path: csh.js
page_number: None
source_type: None
element_index: 0
related_text_ids: []
related_image_ids: []
related_table_ids: []



## Step 8: Statistics and Summary

In [9]:
# Generate statistics
if preserver.all_elements:
    total_elements = len(preserver.all_elements)
    
    # Count relationships
    elements_with_relationships = 0
    total_text_links = 0
    total_image_links = 0
    total_table_links = 0
    
    for elem in preserver.all_elements:
        text_ids = elem.get('related_text_ids', [])
        image_ids = elem.get('related_image_ids', [])
        table_ids = elem.get('related_table_ids', [])
        
        if text_ids or image_ids or table_ids:
            elements_with_relationships += 1
        
        total_text_links += len(text_ids)
        total_image_links += len(image_ids)
        total_table_links += len(table_ids)
    
    print("\n" + "=" * 80)
    print("CONTEXT LINKING STATISTICS")
    print("=" * 80)
    print(f"\nTotal elements processed: {total_elements}")
    print(f"Elements with relationships: {elements_with_relationships} ({elements_with_relationships/total_elements*100:.1f}%)")
    print(f"\nRelationship links:")
    print(f"  Text-to-text: {total_text_links}")
    print(f"  Text-to-image: {total_image_links}")
    print(f"  Text-to-table: {total_table_links}")
    print(f"  Total links: {total_text_links + total_image_links + total_table_links}")
    print(f"\nAverage links per element: {(total_text_links + total_image_links + total_table_links) / total_elements:.2f}")
    print("\n" + "=" * 80)
    
    print("\n✓ Context linking demonstration complete!")
    print("\nNext: 03_summarization.ipynb - AI-powered content summarization")


CONTEXT LINKING STATISTICS

Total elements processed: 123310
Elements with relationships: 91330 (74.1%)

Relationship links:
  Text-to-text: 449212
  Text-to-image: 13655
  Text-to-table: 0
  Total links: 462867

Average links per element: 3.75


✓ Context linking demonstration complete!

Next: 03_summarization.ipynb - AI-powered content summarization
